# Clase 6: Visión Artificial e Inspección Industrial de Metales

En esta sesión damos el salto más importante del curso: pasar de **tablas de números** a **imágenes**. Vamos a entrenar una red neuronal que aprende a detectar defectos en chapas de acero — una tarea que en la industria real se ejecuta a decenas de metros por minuto, donde el ojo humano no puede seguir el ritmo.

---

## ¿Por qué este problema?

Imaginá una línea de laminado de acero. La chapa sale del horno a más de 1000°C y se desplaza a velocidades que hacen imposible la inspección visual humana. Un defecto no detectado puede:
- Llegar al cliente y causar fallas estructurales
- Parar toda la línea de producción para un recall
- Costar millones en garantías y reputación

La solución: una cámara industrial + una red neuronal entrenada para ver lo que nosotros no podemos.

---

## El dataset: NEU Surface Defect Database

Fue creado por la Universidad del Noreste de China (Northeastern University) específicamente para investigación en visión industrial. Es el **benchmark estándar** en este dominio.

**Composición:** 1.800 imágenes de microscopía de luz, divididas en:
- 1.440 para entrenamiento (240 por clase)
- 360 para validación (60 por clase)

**Las 6 categorías de defectos:**

| Defecto | ¿Qué es? | Aspecto visual |
|---|---|---|
| **Crazing** | Red de grietas finas | Patrón de "piel de cocodrilo" |
| **Inclusion** | Impurezas no metálicas atrapadas | Manchas oscuras irregulares |
| **Patches** | Zonas con textura/color anómalo | Áreas más claras u oscuras |
| **Pitted Surface** | Pequeños pozos o porosidad | Puntos oscuros dispersos |
| **Rolled-in Scale** | Óxido incrustado durante laminado | Escamas lineales |
| **Scratches** | Marcas lineales por fricción | Líneas rectas o curvas |

---

## Concepto clave: ¿Qué es una imagen para una red neuronal?

El primer obstáculo conceptual: **una red neuronal no "ve" una imagen, procesa números.**

Una imagen en escala de grises es simplemente una **matriz 2D** donde cada celda contiene la intensidad de ese pixel:
- `0` = negro absoluto
- `255` = blanco absoluto
- Los valores intermedios = grises

```
Imagen 4×4 (simplificada):
┌─────┬─────┬─────┬─────┐
│  12 │  45 │  89 │ 201 │
├─────┼─────┼─────┼─────┤
│  34 │ 120 │  67 │  90 │
├─────┼─────┼─────┼─────┤
│ 200 │  78 │  23 │  45 │
├─────┼─────┼─────┼─────┤
│  56 │  89 │ 178 │  12 │
└─────┴─────┴─────┴─────┘
```

En PyTorch, organizamos estas matrices en **tensores 4D** con el formato `[Batch, Canales, Alto, Ancho]`:
- **Batch**: cuántas imágenes procesamos juntas (ej: 32)
- **Canales**: 1 para gris, 3 para RGB
- **Alto × Ancho**: resolución de la imagen (usaremos 224×224)

Entonces, un batch de 32 imágenes en gris de 224×224 ocupa el tensor: `[32, 1, 224, 224]` = 1.605.632 números.

---

## Concepto clave: ¿Por qué una CNN y no una red común?

Una red densa (MLP) trataría cada pixel como un dato independiente, sin considerar que los pixels **vecinos están relacionados**. Un rasguño es una secuencia de pixels oscuros *en línea*, no pixels oscuros aislados.

Las **Redes Neuronales Convolucionales (CNN)** resuelven esto con tres tipos de capas:

### 1. Capa Convolucional (`Conv2d`)
Desliza un **filtro** (una pequeña matriz, ej: 3×3) por toda la imagen, multiplicando y sumando. Cada filtro aprende a detectar un patrón específico:
- Un filtro puede aprender a detectar bordes horizontales
- Otro, bordes verticales
- En capas más profundas: texturas, formas, defectos completos

La red aprende los valores de estos filtros **sola**, durante el entrenamiento.

### 2. Capa de Pooling (`MaxPool2d`)
Toma cada zona de 2×2 pixels y se queda solo con el valor máximo. Resultado: la imagen se reduce a la mitad. Esto hace que la red sea **tolerante a pequeñas variaciones de posición** (si el rasguño está 2 pixels más a la izquierda, sigue detectándose).

### 3. Capas Densas (`Linear`)
Después de varias rondas de Conv+Pool, "aplastamos" el tensor resultante y lo pasamos por capas densas que toman la decisión final: ¿a cuál de las 6 clases pertenece esta imagen?

---

## Bloque 0: Configuración de reproducibilidad

**¿Por qué existe este bloque?** Las redes neuronales usan números aleatorios durante la inicialización y el entrenamiento. Sin fijar una semilla (`seed`), cada vez que ejecutés el notebook obtendrás resultados ligeramente distintos, lo que hace imposible comparar experimentos o depurar errores. Este es un **estándar profesional** en ciencia de datos.

In [ ]:
import random
import numpy as np
import torch

SEED = 42

def set_seed(seed: int):
    """Fija la semilla aleatoria en todos los módulos relevantes."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # Hace que cuDNN use algoritmos deterministas (puede reducir velocidad levemente)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Semilla fijada: {SEED}")
print(f"PyTorch versión: {torch.__version__}")

## Bloque 1: Adquisición y organización del dataset

**Ejecutar este bloque primero.** Descarga el dataset desde el repositorio de la cátedra y organiza las carpetas para que PyTorch pueda leerlas automáticamente.

La estructura esperada es:
```
data/NEU/
├── train/
│   ├── crazing/      (240 imágenes)
│   ├── inclusion/    (240 imágenes)
│   ├── patches/      (240 imágenes)
│   ├── pitted_surface/ (240 imágenes)
│   ├── rolled-in_scale/ (240 imágenes)
│   └── scratches/    (240 imágenes)
└── validation/
    └── (mismas 6 carpetas, 60 imágenes cada una)
```

PyTorch leerá los **nombres de carpeta** como nombres de clase automáticamente.

In [ ]:
import os
import zipfile
import urllib.request
import shutil

# ── Configuración ────────────────────────────────────────────────────────────
DATA_DIR      = './data'
DATASET_PATH  = os.path.join(DATA_DIR, 'NEU')
ZIP_NAME      = 'IA_CAT_DATA.zip'
REPO_URL      = 'https://github.com/hectorpyco/IA_2026_ATY/archive/refs/heads/main.zip'

os.makedirs(DATA_DIR, exist_ok=True)

# ── Descarga (solo si no existe el zip) ──────────────────────────────────────
if not os.path.exists(ZIP_NAME):
    print('Descargando dataset desde el repositorio de la cátedra...')
    headers = {'User-Agent': 'Mozilla/5.0'}
    req = urllib.request.Request(REPO_URL, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=60) as response, \
             open(ZIP_NAME, 'wb') as out_file:
            shutil.copyfileobj(response, out_file)
        print('Descarga completada.')
    except Exception as e:
        print(f'Error en la descarga: {e}')
        print('Verificá tu conexión a internet e intentá de nuevo.')
        raise
else:
    print('ZIP ya existe, omitiendo descarga.')

# ── Extracción (solo si no existe la carpeta del dataset) ────────────────────
if not os.path.exists(DATASET_PATH):
    print('Extrayendo y normalizando estructura de carpetas...')
    with zipfile.ZipFile(ZIP_NAME, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)

    # GitHub genera un zip con una carpeta raíz del tipo 'repo-main'
    source_folder = os.path.join(DATA_DIR, 'IA_2026_ATY-main', 'NEU_Data')

    if os.path.exists(source_folder):
        shutil.move(source_folder, DATASET_PATH)
        # Limpieza de archivos temporales
        shutil.rmtree(os.path.join(DATA_DIR, 'IA_2026_ATY-main'), ignore_errors=True)
        print(f'Dataset listo en: {DATASET_PATH}')
    else:
        print('ERROR: No se encontró la carpeta NEU_Data dentro del ZIP.')
        print('Contenido extraído:', os.listdir(DATA_DIR))
        raise FileNotFoundError('Estructura del ZIP inesperada.')
else:
    print('Dataset ya existe, omitiendo extracción.')

# ── Verificación ─────────────────────────────────────────────────────────────
print('\nVerificando estructura del dataset:')
for split in ['train', 'validation']:
    split_path = os.path.join(DATASET_PATH, split)
    if os.path.exists(split_path):
        clases = sorted(os.listdir(split_path))
        total  = sum(len(os.listdir(os.path.join(split_path, c))) for c in clases)
        print(f'  {split}: {len(clases)} clases, {total} imágenes')
    else:
        print(f'  ADVERTENCIA: No se encontró la carpeta "{split}"')

## Bloque 2: Exploración visual del dataset

Antes de entrenar cualquier modelo, un buen ingeniero **mira los datos**. Esto te permite detectar:
- ¿Las imágenes tienen la calidad esperada?
- ¿Las clases se ven visualmente distintas?
- ¿Hay imágenes corruptas o mal etiquetadas?

Nunca entrenés un modelo sobre datos que no viste.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

TRAIN_PATH = os.path.join(DATASET_PATH, 'train')
clases = sorted(os.listdir(TRAIN_PATH))

print('Clases encontradas:', clases)

# Mostrar una imagen de muestra de cada clase
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('Muestra de cada tipo de defecto — Dataset NEU', fontsize=14, fontweight='bold')

for ax, clase in zip(axes.flat, clases):
    clase_path = os.path.join(TRAIN_PATH, clase)
    # Tomamos la primera imagen de cada carpeta
    imagen_archivo = sorted(os.listdir(clase_path))[0]
    imagen = Image.open(os.path.join(clase_path, imagen_archivo)).convert('L')
    
    ax.imshow(imagen, cmap='gray')
    ax.set_title(clase.replace('_', ' ').title(), fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Estadísticas del dataset
print('\nDistribución del dataset:')
print(f'{"Clase":<25} {"Train":>8} {"Val":>8}')
print('-' * 43)
for clase in clases:
    n_train = len(os.listdir(os.path.join(DATASET_PATH, 'train', clase)))
    n_val   = len(os.listdir(os.path.join(DATASET_PATH, 'validation', clase)))
    print(f'{clase:<25} {n_train:>8} {n_val:>8}')

## Bloque 3: Preprocesamiento — Transformar imágenes en tensores

Las redes neuronales no trabajan con archivos JPEG; trabajan con tensores de números. Este bloque define la **tubería de transformación** que convierte cada imagen al formato que espera el modelo.

### Data Augmentation — ¿por qué aumentar los datos?

Con solo 240 imágenes por clase para entrenamiento, hay riesgo de **overfitting**: el modelo memoriza las imágenes exactas en vez de aprender el patrón general del defecto.

La solución es **aumentación de datos**: aplicar transformaciones aleatorias a cada imagen en cada época, generando variantes artificiales. La red nunca ve la misma imagen dos veces exactamente igual.

**Transformaciones elegidas:**
- `RandomHorizontalFlip`: un rasguño sigue siendo rasguño si está espejado
- `RandomVerticalFlip`: ídem
- `RandomRotation(15°)`: la chapa puede entrar a la cámara levemente rotada
- `ColorJitter`: simula variaciones de iluminación industrial

**Importante:** La aumentación se aplica **solo al set de entrenamiento**. El set de validación recibe siempre la imagen original, porque queremos medir el rendimiento en condiciones reales.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

IMG_SIZE   = 224
BATCH_SIZE = 32

# ── Transformaciones de entrenamiento (CON aumentación) ───────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    # Aumentación de datos
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    # Conversión y normalización
    transforms.ToTensor(),
    # Normalización: (pixel - media) / desvío → valores en rango ~[-1, 1]
    # Los valores (0.5, 0.5) son una aproximación para imágenes en gris
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

# ── Transformaciones de validación (SIN aumentación) ─────────────────────────
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

# ── Carga de datasets ─────────────────────────────────────────────────────────
# ImageFolder lee automáticamente los nombres de carpeta como clases
train_dataset = datasets.ImageFolder(
    root=os.path.join(DATASET_PATH, 'train'),
    transform=train_transform
)
val_dataset = datasets.ImageFolder(
    root=os.path.join(DATASET_PATH, 'validation'),
    transform=val_transform
)

# ── DataLoaders: la "tubería" que alimenta la GPU ─────────────────────────────
# num_workers: subprocesos que cargan imágenes en paralelo mientras la GPU entrena
# pin_memory: optimización para transferir datos a la GPU más rápido
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          # Mezclar en cada época evita que la red aprenda el orden
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,         # No mezclar: queremos resultados reproducibles
    num_workers=2,
    pin_memory=True
)

CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f'Clases ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Imágenes de entrenamiento: {len(train_dataset)}')
print(f'Imágenes de validación:    {len(val_dataset)}')

# Verificar shape de un batch
imagenes, etiquetas = next(iter(train_loader))
print(f'\nShape de un batch: {imagenes.shape}  → [Batch, Canales, Alto, Ancho]')
print(f'Rango de valores: [{imagenes.min():.2f}, {imagenes.max():.2f}]')

## Bloque 4: Arquitectura de la CNN

### ¿Cómo diseñamos el modelo?

Vamos a construir una CNN desde cero para entender cada capa. La arquitectura tiene 3 bloques convolucionales seguidos de capas densas.

**¿Por qué 3 bloques y no 2 o 5?** Es un balance entre:
- **Capacidad**: más bloques = detecta patrones más complejos
- **Datos disponibles**: con pocos datos, redes muy profundas hacen overfitting
- **Tiempo de entrenamiento**: más capas = más lento

### Cálculo del tamaño tras el pooling

Cada `MaxPool2d(2,2)` divide el tamaño a la mitad:
```
224 × 224  →  (conv1 + pool)  →  112 × 112
112 × 112  →  (conv2 + pool)  →   56 × 56
 56 × 56   →  (conv3 + pool)  →   28 × 28
```
Después de 3 poolings, tenemos 64 filtros de 28×28 → 64 × 28 × 28 = **50.176 valores** para las capas densas.

### Dropout — regularización para evitar overfitting

`Dropout(p=0.5)` apaga aleatoriamente el 50% de las neuronas durante cada paso de entrenamiento. Esto obliga a la red a **no depender de ninguna neurona individual**: aprende representaciones más robustas y generaliza mejor a datos nuevos. Durante la evaluación, el dropout se desactiva automáticamente.

In [ ]:
class NEU_Classifier(nn.Module):
    """
    CNN para clasificación de defectos superficiales en acero.
    
    Arquitectura:
        3 bloques Conv→BN→ReLU→Pool + 2 capas densas con Dropout
    
    Flujo del tensor (ejemplo con batch_size=32):
        Entrada:   [32,  1, 224, 224]
        Tras bloque 1: [32, 16, 112, 112]
        Tras bloque 2: [32, 32,  56,  56]
        Tras bloque 3: [32, 64,  28,  28]
        Flatten:   [32, 50176]
        fc1:       [32, 256]
        fc2:       [32, 6]    ← una puntuación por clase
    """
    def __init__(self, num_classes: int = 6):
        super().__init__()
        
        # ── Bloque 1: detecta bordes y texturas simples ──────────────────────
        self.bloque1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),   # Estabiliza el entrenamiento normalizando activaciones
            nn.ReLU(),
            nn.MaxPool2d(2, 2)    # 224 → 112
        )
        
        # ── Bloque 2: combina texturas para detectar patrones más complejos ──
        self.bloque2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)    # 112 → 56
        )
        
        # ── Bloque 3: detecta defectos completos ─────────────────────────────
        self.bloque3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)    # 56 → 28
        )
        
        # ── Clasificador denso ────────────────────────────────────────────────
        self.clasificador = nn.Sequential(
            nn.Flatten(),                        # [B, 64, 28, 28] → [B, 50176]
            nn.Linear(64 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(p=0.5),                   # Regularización
            nn.Linear(256, num_classes)          # Salida: 6 puntuaciones (logits)
        )
    
    def forward(self, x):
        x = self.bloque1(x)
        x = self.bloque2(x)
        x = self.bloque3(x)
        x = self.clasificador(x)
        return x


# ── Inicialización y envío a GPU ──────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = NEU_Classifier(num_classes=NUM_CLASSES).to(device)

if torch.cuda.is_available():
    print(f'GPU activa: {torch.cuda.get_device_name(0)}')
    print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ADVERTENCIA: No se detectó GPU. El entrenamiento será lento en CPU.')

# ── Resumen del modelo ────────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
params_entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParámetros totales:       {total_params:,}')
print(f'Parámetros entrenables:   {params_entrenables:,}')
print('\nArquitectura:')
print(model)

## Bloque 5: Entrenamiento

### Los ingredientes del aprendizaje

**Función de pérdida (`CrossEntropyLoss`):** Mide qué tan equivocado está el modelo. Para clasificación multiclase es la elección estándar: combina `Softmax` (convierte logits en probabilidades) con `NegativeLogLikelihood` (penaliza más cuando el modelo está muy seguro de la respuesta incorrecta).

**Optimizador (`Adam`):** Ajusta los pesos del modelo para reducir la pérdida. Adam es la variante más usada en práctica: adapta la tasa de aprendizaje para cada parámetro individualmente.

**Learning Rate Scheduler:** Reduce la tasa de aprendizaje cuando la pérdida de validación deja de mejorar. Es como afinar un instrumento: al principio, ajustes grandes; cerca del óptimo, ajustes finos.

**Early Stopping:** Detiene el entrenamiento si la pérdida de validación no mejora en N épocas consecutivas. Evita que el modelo haga overfitting sobreentrenándose en el set de train.

In [ ]:
# ── Hiperparámetros ───────────────────────────────────────────────────────────
EPOCHS         = 20
LEARNING_RATE  = 1e-3
PATIENCE       = 5     # Early stopping: épocas sin mejora antes de detener

# ── Función de pérdida, optimizador y scheduler ───────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
# ReduceLROnPlateau: si val_loss no mejora en 3 épocas, lr = lr × 0.5
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, verbose=True
)

# ── Historial para graficar curvas de aprendizaje ────────────────────────────
historial = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [],  'val_acc': []
}

# ── Early stopping (variables de control) ────────────────────────────────────
mejor_val_loss   = float('inf')
epocas_sin_mejora = 0
RUTA_MEJOR_MODELO = 'mejor_modelo_neu.pth'


def evaluar(loader):
    """Evalúa el modelo en un DataLoader. Retorna (pérdida_promedio, accuracy)."""
    model.eval()
    total_loss, correctos, total = 0.0, 0, 0
    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            outputs = model(imagenes)
            loss    = criterion(outputs, etiquetas)
            total_loss += loss.item()
            _, predicciones = torch.max(outputs, 1)
            correctos += (predicciones == etiquetas).sum().item()
            total     += etiquetas.size(0)
    return total_loss / len(loader), correctos / total


# ── Ciclo de entrenamiento ────────────────────────────────────────────────────
print('Iniciando entrenamiento...')
print(f'{"Época":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>9} | {"Val Acc":>8}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    # ── Fase de entrenamiento ─────────────────────────────────────────────
    model.train()
    train_loss_acc = 0.0
    train_correctos, train_total = 0, 0
    
    for imagenes, etiquetas in train_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        
        optimizer.zero_grad()              # 1. Limpiar gradientes del paso anterior
        outputs = model(imagenes)          # 2. Forward pass
        loss    = criterion(outputs, etiquetas)  # 3. Calcular pérdida
        loss.backward()                    # 4. Backpropagation
        
        # Gradient clipping: evita que los gradientes exploten en valores gigantes
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()                   # 5. Actualizar pesos
        
        train_loss_acc += loss.item()
        _, preds = torch.max(outputs, 1)
        train_correctos += (preds == etiquetas).sum().item()
        train_total     += etiquetas.size(0)
    
    train_loss = train_loss_acc / len(train_loader)
    train_acc  = train_correctos / train_total
    
    # ── Fase de validación ────────────────────────────────────────────────
    val_loss, val_acc = evaluar(val_loader)
    
    # Actualizar scheduler con la pérdida de validación
    scheduler.step(val_loss)
    
    # Guardar historial
    historial['train_loss'].append(train_loss)
    historial['val_loss'].append(val_loss)
    historial['train_acc'].append(train_acc)
    historial['val_acc'].append(val_acc)
    
    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.1%} | {val_loss:>9.4f} | {val_acc:>7.1%}')
    
    # ── Early stopping y guardado del mejor modelo ────────────────────────
    if val_loss < mejor_val_loss:
        mejor_val_loss    = val_loss
        epocas_sin_mejora = 0
        torch.save(model.state_dict(), RUTA_MEJOR_MODELO)
        print(f'         Mejor modelo guardado (val_loss={val_loss:.4f})')
    else:
        epocas_sin_mejora += 1
        if epocas_sin_mejora >= PATIENCE:
            print(f'\nEarly stopping: sin mejora en {PATIENCE} épocas consecutivas.')
            break

print(f'\nEntrenamiento finalizado. Mejor val_loss: {mejor_val_loss:.4f}')

## Bloque 6: Curvas de aprendizaje

Las curvas de aprendizaje son el **diagnóstico principal** de un modelo. Permiten detectar:

- **Underfitting**: ambas curvas (train y val) tienen pérdida alta → el modelo es muy simple o entrenó muy poco
- **Overfitting**: la pérdida de train baja pero la de val sube o se estanca → el modelo memorizó el train
- **Buen entrenamiento**: ambas curvas bajan juntas y convergen

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epocas = range(1, len(historial['train_loss']) + 1)

# ── Gráfico de pérdida ────────────────────────────────────────────────────────
ax1.plot(epocas, historial['train_loss'], 'b-o', markersize=4, label='Train')
ax1.plot(epocas, historial['val_loss'],   'r-o', markersize=4, label='Validación')
ax1.set_title('Pérdida (Loss) por época')
ax1.set_xlabel('Época')
ax1.set_ylabel('CrossEntropyLoss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Gráfico de accuracy ───────────────────────────────────────────────────────
ax2.plot(epocas, [a*100 for a in historial['train_acc']], 'b-o', markersize=4, label='Train')
ax2.plot(epocas, [a*100 for a in historial['val_acc']],   'r-o', markersize=4, label='Validación')
ax2.set_title('Precisión (Accuracy) por época')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim(0, 105)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Curvas de Aprendizaje — NEU Classifier', fontsize=13)
plt.tight_layout()
plt.show()

print('Interpretación guiada:')
print(f'  Accuracy final entrenamiento: {historial["train_acc"][-1]:.1%}')
print(f'  Accuracy final validación:    {historial["val_acc"][-1]:.1%}')
gap = historial['train_acc'][-1] - historial['val_acc'][-1]
if gap > 0.15:
    print(f'  Brecha train/val: {gap:.1%} → posible overfitting')
elif historial['val_acc'][-1] < 0.7:
    print(f'  Accuracy de validación baja → posible underfitting')
else:
    print(f'  Brecha train/val: {gap:.1%} → entrenamiento razonable')

## Bloque 7: Evaluación — Matriz de Confusión y Métricas

### ¿Por qué el accuracy solo no es suficiente?

Si el modelo acierta el 90% de las veces en promedio, ¿está bien? **Depende.** Si comete el 100% de errores en una sola clase (por ejemplo, siempre confunde *Crazing* con *Scratches*), eso es un problema grave en producción aunque el accuracy global sea alto.

**Las métricas correctas son:**
- **Precision**: de todo lo que el modelo dijo "es clase X", ¿cuánto era realmente clase X? → penaliza falsos positivos
- **Recall**: de todo lo que era realmente clase X, ¿cuánto detectó el modelo? → penaliza falsos negativos
- **F1-Score**: media armónica de Precision y Recall → el balance entre ambos

**En inspección industrial, el Recall es crítico:** es más grave dejar pasar un defecto (falso negativo) que marcar una chapa buena como defectuosa (falso positivo). Una chapa defectuosa que llega al cliente puede causar fallas estructurales; una chapa buena descartada solo es un costo extra.

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ── Cargar el mejor modelo guardado ──────────────────────────────────────────
model.load_state_dict(torch.load(RUTA_MEJOR_MODELO, map_location=device))
print(f'Modelo cargado desde: {RUTA_MEJOR_MODELO}')

# ── Recolectar predicciones sobre todo el set de validación ──────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imagenes, etiquetas in val_loader:
        imagenes = imagenes.to(device)
        outputs  = model(imagenes)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(etiquetas.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# ── Matriz de confusión ───────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    linewidths=0.5
)
plt.xlabel('Predicción del modelo', fontsize=12)
plt.ylabel('Clase real (verdad)', fontsize=12)
plt.title('Matriz de Confusión — Defectos de Acero NEU', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ── Reporte completo ──────────────────────────────────────────────────────────
print('Reporte de Clasificación:\n')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# ── Interpretación automática: clases problemáticas ──────────────────────────
print('Clases con menor F1-Score (más difíciles de clasificar):')
report_dict = {}
from sklearn.metrics import f1_score
f1s = f1_score(all_labels, all_preds, average=None)
for clase, f1 in sorted(zip(CLASS_NAMES, f1s), key=lambda x: x[1]):
    print(f'  {clase:<25} F1 = {f1:.3f}')

## Bloque 8: Visualización de errores del modelo

Ver **qué imágenes clasificó mal** el modelo es igual de valioso que ver las métricas. Permite entender si los errores son "razonables" (confundir *Crazing* con *Inclusion* puede tener sentido visualmente) o si hay problemas sistemáticos (el modelo nunca detecta correctamente una clase específica).

In [ ]:
# Encontrar índices de errores
indices_error = np.where(all_preds != all_labels)[0]

print(f'Total de errores: {len(indices_error)} de {len(all_labels)} imágenes')
print(f'Accuracy en validación: {(all_preds == all_labels).mean():.1%}')

if len(indices_error) == 0:
    print('¡Clasificación perfecta! No hay errores para mostrar.')
else:
    # Mostrar hasta 8 ejemplos de errores
    n_mostrar = min(8, len(indices_error))
    muestra_errores = np.random.choice(indices_error, n_mostrar, replace=False)
    
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    fig.suptitle('Ejemplos de errores del modelo', fontsize=13)
    
    # Necesitamos recuperar las imágenes originales del val_dataset
    for ax, idx in zip(axes.flat, muestra_errores):
        imagen, _ = val_dataset[idx]
        imagen_np = imagen.squeeze().numpy()  # [1, H, W] → [H, W]
        # Desnormalizar para visualización
        imagen_np = imagen_np * 0.5 + 0.5
        
        real = CLASS_NAMES[all_labels[idx]]
        pred = CLASS_NAMES[all_preds[idx]]
        
        ax.imshow(imagen_np, cmap='gray')
        ax.set_title(f'Real: {real}\nPred: {pred}', fontsize=9,
                     color='red' if real != pred else 'green')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

## Bloque 9: Inferencia — Prueba con una imagen individual

Este bloque simula cómo funcionaría el sistema en producción: recibe una imagen, la procesa y devuelve la predicción con el nivel de confianza del modelo.

La **confianza** se obtiene aplicando `Softmax` a los logits de salida: convierte los 6 números de salida en probabilidades que suman 1.

In [ ]:
def predecir_imagen(ruta_imagen: str, modelo, transformacion, nombres_clase, dispositivo):
    """
    Clasifica una imagen y visualiza el resultado con el nivel de confianza.
    
    Args:
        ruta_imagen: Ruta al archivo de imagen
        modelo: Modelo PyTorch entrenado
        transformacion: Pipeline de transformaciones (val_transform)
        nombres_clase: Lista de nombres de clase
        dispositivo: 'cuda' o 'cpu'
    """
    # Cargar y preprocesar
    imagen_pil = Image.open(ruta_imagen).convert('L')
    tensor = transformacion(imagen_pil).unsqueeze(0).to(dispositivo)  # [1, 1, 224, 224]
    
    # Inferencia
    modelo.eval()
    with torch.no_grad():
        logits = modelo(tensor)                            # [1, 6] valores crudos
        probs  = F.softmax(logits, dim=1).squeeze()        # [6] probabilidades
        conf, pred_idx = torch.max(probs, 0)
    
    clase_pred = nombres_clase[pred_idx.item()]
    confianza  = conf.item()
    
    # Visualización
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Imagen con predicción
    ax1.imshow(imagen_pil, cmap='gray')
    color = 'green' if confianza > 0.8 else 'orange'
    ax1.set_title(f'Predicción: {clase_pred}\nConfianza: {confianza:.1%}',
                  fontsize=12, color=color, fontweight='bold')
    ax1.axis('off')
    
    # Gráfico de barras con todas las probabilidades
    probs_np = probs.cpu().numpy()
    colores  = ['steelblue'] * len(nombres_clase)
    colores[pred_idx.item()] = 'crimson'
    
    bars = ax2.barh(nombres_clase, probs_np * 100, color=colores)
    ax2.set_xlabel('Confianza (%)')
    ax2.set_title('Distribución de probabilidades')
    ax2.set_xlim(0, 105)
    for bar, prob in zip(bars, probs_np):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{prob:.1%}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    return clase_pred, confianza


# ── Ejemplo de uso ────────────────────────────────────────────────────────────
# Modificá la ruta para probar con cualquier imagen
ruta_prueba = f'./data/NEU/validation/{CLASS_NAMES[0]}/{CLASS_NAMES[0]}_275.jpg'

if os.path.exists(ruta_prueba):
    clase, confianza = predecir_imagen(
        ruta_prueba, model, val_transform, CLASS_NAMES, device
    )
    print(f'Resultado: {clase} (confianza: {confianza:.1%})')
else:
    print(f'No se encontró el archivo: {ruta_prueba}')
    print('Modificá la variable ruta_prueba con una imagen válida.')

## Bloque 10: Guardar el modelo

En PyTorch guardamos el **`state_dict`**: un diccionario con todos los pesos del modelo. No guardamos el modelo completo porque la arquitectura ya está definida en el código.

Para cargar el modelo en otro script o notebook:
```python
model = NEU_Classifier(num_classes=6)
model.load_state_dict(torch.load('modelo_neu_clase6.pth'))
model.eval()
```

In [ ]:
import json
from datetime import datetime

RUTA_MODELO_FINAL = 'modelo_neu_clase6.pth'

# Guardar pesos del mejor modelo
torch.save(model.state_dict(), RUTA_MODELO_FINAL)

# Guardar metadatos del experimento (buena práctica profesional)
metadatos = {
    'fecha': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'clases': CLASS_NAMES,
    'num_clases': NUM_CLASSES,
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'epocas_entrenadas': len(historial['train_loss']),
    'mejor_val_loss': round(mejor_val_loss, 4),
    'val_accuracy_final': round(historial['val_acc'][-1], 4),
    'seed': SEED
}

with open('metadatos_modelo.json', 'w') as f:
    json.dump(metadatos, f, indent=2)

print(f'Modelo guardado en: {RUTA_MODELO_FINAL}')
print(f'Metadatos guardados en: metadatos_modelo.json')
print(f'\nResumen del experimento:')
for k, v in metadatos.items():
    print(f'  {k}: {v}')

---

# Tarea: El Desafío del Inspector

El éxito de un modelo de IA se mide cuando sale del ambiente controlado del notebook y enfrenta el "mundo real".

## Paso A: Validación externa

1. Buscá en Google Imágenes o bases de datos metalúrgicas fotos de `"Steel surface defects"` que **no** pertenezcan al dataset NEU.
2. Descargá al menos 3 imágenes de distintos tipos de defecto.
3. Guardalas en tu carpeta de trabajo.
4. Usá la función `predecir_imagen()` del Bloque 9 para clasificarlas.

## Paso B: Análisis crítico (para el informe)

Respondé las siguientes preguntas con evidencia de tus experimentos:

**1. Confianza en imágenes externas:** ¿El modelo mantiene una confianza alta (>80%) en imágenes de internet? Mostrá la distribución de probabilidades de al menos 3 imágenes externas.

**2. Dominio de distribución:** El dataset NEU fue capturado con un equipo específico de microscopía. ¿Qué factores crees que causan que el modelo se confunda con fotos de internet? Considerá: iluminación, zoom, resolución, ángulo de captura, artefactos de compresión JPEG.

**3. Análisis de la matriz de confusión:** ¿Qué pares de clases confunde más frecuentemente el modelo? ¿Podés explicar por qué visualmente?

**4. (Desafío opcional):** El dataset NEU tiene exactamente el mismo número de imágenes por clase (240 train / 60 val). En la industria real, los defectos **no** están balanceados: algunos son raros y peligrosos. ¿Qué técnica usarías para manejar un dataset desbalanceado? Investigá `WeightedRandomSampler` en PyTorch.

---

## Conceptos clave de esta clase

| Concepto | ¿Qué es? | ¿Por qué importa? |
|---|---|---|
| **Tensor 4D** | `[Batch, Canal, H, W]` | Formato estándar de imágenes en PyTorch |
| **Convolución** | Filtro que detecta patrones locales | El núcleo de las CNN |
| **BatchNorm** | Normaliza activaciones por batch | Estabiliza y acelera el entrenamiento |
| **Dropout** | Apaga neuronas aleatoriamente | Evita overfitting |
| **Data Augmentation** | Transforma imágenes aleatoriamente | Más datos virtuales, mejor generalización |
| **Early Stopping** | Detiene si val_loss no mejora | Previene overfitting |
| **F1-Score** | Balance entre precision y recall | Métrica correcta para inspección industrial |
| **Domain Shift** | Diferencia entre datos de train y reales | El mayor problema del ML en producción |